In [1]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

In [2]:
import pandas as pd
from src.dataloaders import SeasonAverageDataLoader
from src.experiments import ExperimentConfig, DefaultTracker
from src.models.xgboost import XGBRegressorModel, XGBHyperparamConfig, CrossValidationConfig, get_run_name

#Train model on all features

In [3]:
features = [
    "MenWomen",
    "QualityDiff",
    "Season",
    "SeedDiff",
    "T1_Elo",
    "T1_Quality",
    "T1_TeamID",
    "T1_avg_Ast",
    "T1_avg_Blk",
    "T1_avg_DR",
    "T1_avg_Elo",
    "T1_avg_EloDelta",
    "T1_avg_EloDeltaDiff",
    "T1_avg_EloDiff",
    "T1_avg_FGA",
    "T1_avg_FGA3",
    "T1_avg_FGM",
    "T1_avg_FGM3",
    "T1_avg_FTA",
    "T1_avg_FTM",
    "T1_avg_OR",
    "T1_avg_PF",
    "T1_avg_PointDiff",
    "T1_avg_Score",
    "T1_avg_Stl",
    "T1_avg_Streak",
    "T1_avg_StreakDiff",
    "T1_avg_TO",
    "T1_avg_opponent_Ast",
    "T1_avg_opponent_Blk",
    "T1_avg_opponent_DR",
    "T1_avg_opponent_Elo",
    "T1_avg_opponent_EloDelta",
    "T1_avg_opponent_FGA",
    "T1_avg_opponent_FGA3",
    "T1_avg_opponent_FGM",
    "T1_avg_opponent_FGM3",
    "T1_avg_opponent_FTA",
    "T1_avg_opponent_FTM",
    "T1_avg_opponent_OR",
    "T1_avg_opponent_PF",
    "T1_avg_opponent_Score",
    "T1_avg_opponent_Stl",
    "T1_avg_opponent_Streak",
    "T1_avg_opponent_TO",
    "T1_seed",
    "T2_Elo",
    "T2_Quality",
    "T2_TeamID",
    "T2_avg_Ast",
    "T2_avg_Blk",
    "T2_avg_DR",
    "T2_avg_Elo",
    "T2_avg_EloDelta",
    "T2_avg_EloDeltaDiff",
    "T2_avg_EloDiff",
    "T2_avg_FGA",
    "T2_avg_FGA3",
    "T2_avg_FGM",
    "T2_avg_FGM3",
    "T2_avg_FTA",
    "T2_avg_FTM",
    "T2_avg_OR",
    "T2_avg_PF",
    "T2_avg_PointDiff",
    "T2_avg_Score",
    "T2_avg_Stl",
    "T2_avg_Streak",
    "T2_avg_StreakDiff",
    "T2_avg_TO",
    "T2_avg_opponent_Ast",
    "T2_avg_opponent_Blk",
    "T2_avg_opponent_DR",
    "T2_avg_opponent_Elo",
    "T2_avg_opponent_EloDelta",
    "T2_avg_opponent_FGA",
    "T2_avg_opponent_FGA3",
    "T2_avg_opponent_FGM",
    "T2_avg_opponent_FGM3",
    "T2_avg_opponent_FTA",
    "T2_avg_opponent_FTM",
    "T2_avg_opponent_OR",
    "T2_avg_opponent_PF",
    "T2_avg_opponent_Score",
    "T2_avg_opponent_Stl",
    "T2_avg_opponent_Streak",
    "T2_avg_opponent_TO",
    "T2_seed",
]

In [4]:
validation_season = 2024
data_loader = SeasonAverageDataLoader(features)
regressor_config = XGBHyperparamConfig()
experiment_config = ExperimentConfig(name=get_run_name(regressor_config, features, validation_season))
tracker = DefaultTracker(experiment_config)

In [5]:
model = XGBRegressorModel(data_loader, regressor_config, CrossValidationConfig(), tracker)

In [6]:
model.fit(validation_season)

metrics: {'fold': 0, 'train_brier_cv_fold': np.float64(0.17767522170056782)}, step: None
metrics: {'fold': 1, 'train_brier_cv_fold': np.float64(0.16612313485242028)}, step: None
metrics: {'fold': 2, 'train_brier_cv_fold': np.float64(0.17887090101758227)}, step: None
metrics: {'fold': 3, 'train_brier_cv_fold': np.float64(0.1649611956223957)}, step: None
metrics: {'fold': 4, 'train_brier_cv_fold': np.float64(0.17326013781698607)}, step: None
metrics: {'train_brier_cv_full': np.float64(0.172177865629718)}, step: None
metrics: {'train_brier': np.float64(0.08000729331147212)}, step: None


In [7]:
model.validate()

metrics: {'valid_brier': np.float64(0.17274471753437834)}, step: None


## Determine feature importance for model

In [8]:
df_trees = model.model.trees_to_dataframe()

In [9]:
df_trees.head()

,Tree,Node,ID,Feature,Split,Yes,No,Missing,Gain,Cover,Category
0,0,0,0-0,QualityDiff,0.151053,0-1,0-2,0-2,213.405533,3408.0,NaN
1,0,1,0-1,QualityDiff,-7.596179,0-3,0-4,0-4,35.317070,1708.0,NaN
2,0,2,0-2,QualityDiff,8.505849,0-5,0-6,0-6,33.385040,1700.0,NaN
3,0,3,0-3,QualityDiff,-14.433825,0-7,0-8,0-8,5.724030,877.0,NaN
4,0,4,0-4,T1_Elo,1366.610230,0-19,0-20,0-20,3.275032,831.0,NaN


In [10]:
df_trees.drop(columns=["Tree", "Node", "ID", "Yes", "No", "Missing", "Category"], inplace=True)
df_trees.head()

,Feature,Split,Gain,Cover
0,QualityDiff,0.151053,213.405533,3408.0
1,QualityDiff,-7.596179,35.317070,1708.0
2,QualityDiff,8.505849,33.385040,1700.0
3,QualityDiff,-14.433825,5.724030,877.0
4,T1_Elo,1366.610230,3.275032,831.0


In [11]:
df_trees.isna().any()

Feature    False
Split       True
Gain       False
Cover      False
dtype: bool

In [12]:
df_trees[df_trees["Split"].isna()]

,Feature,Split,Gain,Cover
17,Leaf,NaN,0.000122,285.0
18,Leaf,NaN,-0.002020,98.0
27,Leaf,NaN,-0.000000,74.0
28,Leaf,NaN,-0.002255,285.0
30,Leaf,NaN,0.004167,5.0
...,...,...,...,...
40477,Leaf,NaN,-0.000536,7.0
40478,Leaf,NaN,0.000215,3.0
40479,Leaf,NaN,0.000801,2.0
40480,Leaf,NaN,-0.000039,3.0


In [13]:
df_trees.dropna(inplace=True)

In [14]:
df_trees.isna().any()

Feature    False
Split      False
Gain       False
Cover      False
dtype: bool

Remember how often which feature was included -> also an indication of importance

In [15]:
df_value_counts = df_trees["Feature"].value_counts()

We are only interested in the gain, since this tells us how much better the model got by including a given feature in a given tree

In [16]:
df_features = df_trees.groupby(["Feature"])[["Gain"]].agg(["max", "mean", "median"])
df_features.columns = df_features.columns.get_level_values(1)

Max, mean and median gain of each feature should help us quantify an overall feature importance

In [17]:
df_features_median = df_features.sort_values(by="median", ascending=False).reset_index()
df_features_mean = df_features.sort_values(by="mean", ascending=False).reset_index()
df_features_max = df_features.sort_values(by="max", ascending=False).reset_index()

In [18]:
df_features_median.head()

,Feature,max,mean,median
0,T2_avg_FTM,3.171921,1.346031,1.184994
1,T1_avg_FGM3,3.951627,1.268946,1.130612
2,T2_avg_Elo,27.966724,1.417135,1.126167
3,T1_avg_FTM,3.237810,1.202952,1.117035
4,T2_avg_EloDiff,4.308630,1.220110,1.104388


In [19]:
df_features_mean.head()

,Feature,max,mean,median
0,QualityDiff,213.405533,5.335703,0.480226
1,SeedDiff,180.692352,4.373731,0.454921
2,T2_seed,59.875641,2.231619,0.965310
3,T1_seed,78.303917,2.085894,1.082589
4,T2_Elo,95.525375,1.809495,0.971372


In [20]:
df_features_max.head()

,Feature,max,mean,median
0,QualityDiff,213.405533,5.335703,0.480226
1,SeedDiff,180.692352,4.373731,0.454921
2,T2_Elo,95.525375,1.809495,0.971372
3,T1_seed,78.303917,2.085894,1.082589
4,T1_Elo,69.589066,1.346863,0.711234


normalize each statistical column so we can calculate a somewhat interpretable score of feature importance

In [21]:
norm_cols = ["max", "mean", "median"]
df_features_norm = df_features[norm_cols] / df_features[norm_cols].max()
df_features_norm.head()

,max,mean,median
Feature,,,
MenWomen,0.009687,0.045074,0.144717
QualityDiff,1.000000,1.000000,0.405256
SeedDiff,0.846709,0.819710,0.383901
T1_Elo,0.326088,0.252425,0.600200
T1_Quality,0.018918,0.166673,0.608320


In [22]:
df_features_norm = pd.merge(df_features_norm, df_value_counts, on="Feature", how="left")
df_features_norm.head()

,max,mean,median,count
Feature,,,,
MenWomen,0.009687,0.045074,0.144717,94
QualityDiff,1.000000,1.000000,0.405256,2201
SeedDiff,0.846709,0.819710,0.383901,476
T1_Elo,0.326088,0.252425,0.600200,594
T1_Quality,0.018918,0.166673,0.608320,305


create a feature importance score by multiplying each statistical column with the number of times the feature was included in a tree and summing these values up

In [23]:
df_features_norm["score"] = df_features_norm[norm_cols].mul(df_features_norm["count"], axis=0).sum(axis=1)
df_features_norm.head()

,max,mean,median,count,score
Feature,,,,,
MenWomen,0.009687,0.045074,0.144717,94,18.750991
QualityDiff,1.000000,1.000000,0.405256,2201,5293.968849
SeedDiff,0.846709,0.819710,0.383901,476,975.952478
T1_Elo,0.326088,0.252425,0.600200,594,700.155634
T1_Quality,0.018918,0.166673,0.608320,305,242.143069


In [24]:
df_features_scored = df_features_norm.sort_values(by="score", ascending=False).reset_index()
df_features_scored.head()

,Feature,max,mean,median,count,score
0,QualityDiff,1.000000,1.000000,0.405256,2201,5293.968849
1,SeedDiff,0.846709,0.819710,0.383901,476,975.952478
2,T1_Elo,0.326088,0.252425,0.600200,594,700.155634
3,T2_Elo,0.447624,0.339130,0.819727,393,631.346822
4,T1_avg_Elo,0.094708,0.246209,0.852188,289,344.807584


print the scored feature list in a way it can be copied to other code

In [25]:
ranked_features = df_features_scored["Feature"].values.tolist()
print("features = [")
for feature in ranked_features:
    print(f'\t"{feature}",')
print("]")

features = [
	"QualityDiff",
	"SeedDiff",
	"T1_Elo",
	"T2_Elo",
	"T1_avg_Elo",
	"T2_avg_Elo",
	"T1_avg_Blk",
	"T2_avg_Blk",
	"T1_avg_DR",
	"T2_avg_Stl",
	"T1_avg_opponent_OR",
	"T1_avg_Stl",
	"T2_avg_opponent_OR",
	"T1_avg_FGM3",
	"T2_avg_EloDeltaDiff",
	"T1_avg_opponent_Streak",
	"T2_avg_DR",
	"T1_avg_Ast",
	"T1_avg_EloDeltaDiff",
	"T2_avg_FGA",
	"T2_avg_FGM3",
	"T2_avg_opponent_Elo",
	"T1_avg_FGA",
	"T1_Quality",
	"T2_avg_opponent_FGA",
	"T1_avg_opponent_FGA",
	"T1_avg_PF",
	"T2_avg_PF",
	"T2_avg_FGM",
	"T1_avg_opponent_TO",
	"T1_avg_opponent_Ast",
	"T2_avg_opponent_Stl",
	"T2_avg_EloDiff",
	"T1_avg_opponent_Stl",
	"T2_avg_opponent_Streak",
	"T1_avg_FGM",
	"T1_avg_opponent_Blk",
	"T2_avg_opponent_FGA3",
	"T2_avg_opponent_Blk",
	"T2_avg_opponent_Ast",
	"T1_avg_opponent_Elo",
	"T1_avg_opponent_PF",
	"T1_avg_FTM",
	"T2_avg_Ast",
	"T1_avg_Score",
	"T1_avg_opponent_EloDelta",
	"T2_avg_FTM",
	"T1_avg_EloDelta",
	"T2_avg_Score",
	"T1_avg_FTA",
	"T1_avg_opponent_FGA3",
	"T2_avg_opponent_TO",